# Module 01 — Lesson 03: Python Async Primer

This notebook walks through **async/await** in Python with live, runnable examples.

Run each cell in order and observe the output — especially the timing differences between sequential and concurrent execution.

---

> **Note:** Jupyter notebooks already run inside an event loop, so we use
> `nest_asyncio` to allow `asyncio.run()` / `await` at the top level.
> This is only needed in notebooks — regular Python scripts work without it.

In [1]:
# Install nest_asyncio if not already installed
# (only needed for running async code inside Jupyter)
import subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install", "nest_asyncio", "-q"])

import asyncio
import time
import nest_asyncio

nest_asyncio.apply()  # Allow nested event loops in Jupyter
print("✅ Setup complete — asyncio and nest_asyncio ready!")

✅ Setup complete — asyncio and nest_asyncio ready!


---
## Example 1: Sync vs Async Functions

| | Sync | Async |
|---|---|---|
| Definition | `def f():` | `async def f():` |
| Sleep | `time.sleep(n)` | `await asyncio.sleep(n)` |
| Blocks? | ✅ Blocks everything | ❌ Yields control |

Define both versions below:

In [2]:
def sync_task(name: str, seconds: int) -> str:
    """A synchronous function — BLOCKS the entire thread while sleeping."""
    print(f"  [SYNC] {name} starting...")
    time.sleep(seconds)          # ← Blocks everything! No other code runs during this.
    print(f"  [SYNC] {name} finished after {seconds}s")
    return f"{name}_result"


async def async_task(name: str, seconds: int) -> str:
    """An async function — YIELDS control while waiting, letting other tasks run."""
    print(f"  [ASYNC] {name} starting...")
    await asyncio.sleep(seconds) # ← Yields control to the event loop.
    print(f"  [ASYNC] {name} finished after {seconds}s")
    return f"{name}_result"

print("Functions defined ✅")

Functions defined ✅


---
## Example 2a: Sequential Async Execution (Slow) 🐢

When you `await` tasks one after another, they run **in sequence** — each waits for the previous to finish.

Expected time: **~6 seconds** (2s + 2s + 2s)

In [3]:
async def run_sequential():
    """Run three tasks one after another — slow!"""
    print("--- Sequential Execution ---")
    start = time.time()

    result_a = await async_task("Task-A", 2)  # Wait 2s
    result_b = await async_task("Task-B", 2)  # Wait 2s MORE
    result_c = await async_task("Task-C", 2)  # Wait 2s MORE

    elapsed = time.time() - start
    print(f"\n  Results : {result_a}, {result_b}, {result_c}")
    print(f"  ⏱  Total : {elapsed:.1f}s  (expected ~6s)")

asyncio.run(run_sequential())

--- Sequential Execution ---
  [ASYNC] Task-A starting...
  [ASYNC] Task-A finished after 2s
  [ASYNC] Task-B starting...
  [ASYNC] Task-B finished after 2s
  [ASYNC] Task-C starting...
  [ASYNC] Task-C finished after 2s

  Results : Task-A_result, Task-B_result, Task-C_result
  ⏱  Total : 6.0s  (expected ~6s)


---
## Example 2b: Concurrent Async Execution (Fast) 🚀

`asyncio.gather()` launches all tasks **at the same time**. While Task-A is sleeping, Task-B and Task-C are also sleeping — they overlap!

Expected time: **~2 seconds** (all 3 tasks sleep simultaneously)

In [4]:
async def run_concurrent():
    """Run three tasks concurrently using asyncio.gather — fast!"""
    print("--- Concurrent Execution (asyncio.gather) ---")
    start = time.time()

    # All three tasks START immediately and run at the same time
    result_a, result_b, result_c = await asyncio.gather(
        async_task("Task-A", 2),
        async_task("Task-B", 2),
        async_task("Task-C", 2),
    )

    elapsed = time.time() - start
    print(f"\n  Results : {result_a}, {result_b}, {result_c}")
    print(f"  ⏱  Total : {elapsed:.1f}s  (expected ~2s — 3× faster!)")

asyncio.run(run_concurrent())

--- Concurrent Execution (asyncio.gather) ---
  [ASYNC] Task-A starting...
  [ASYNC] Task-B starting...
  [ASYNC] Task-C starting...
  [ASYNC] Task-A finished after 2s
  [ASYNC] Task-B finished after 2s
  [ASYNC] Task-C finished after 2s

  Results : Task-A_result, Task-B_result, Task-C_result
  ⏱  Total : 2.0s  (expected ~2s — 3× faster!)


> **Key insight**: `asyncio.gather()` saved ~4 seconds by running tasks concurrently.
> The more I/O-bound tasks you have (DB queries, HTTP calls), the bigger the speedup!

---
## Example 3: Real-World Pattern — Fetching Multiple Resources

A common scenario: a user profile page needs both the **user record** and **their posts** from the database.

With `gather()`, both queries run at the same time.

In [5]:
async def fetch_user(user_id: int) -> dict:
    """Simulate a database query for a user (takes ~1s)."""
    print(f"  [DB] Fetching user {user_id}...")
    await asyncio.sleep(1)   # Simulates DB latency
    return {"id": user_id, "name": f"User_{user_id}"}


async def fetch_user_posts(user_id: int) -> list:
    """Simulate a database query for posts (takes ~1.5s)."""
    print(f"  [DB] Fetching posts for user {user_id}...")
    await asyncio.sleep(1.5)  # Slightly slower query
    return [
        {"id": 1, "title": f"Post by User_{user_id}"},
        {"id": 2, "title": f"Another post by User_{user_id}"},
    ]


async def fetch_user_profile(user_id: int) -> dict:
    """Fetch user + posts concurrently — total time = max(1, 1.5) = ~1.5s."""
    print(f"--- Fetching profile for user {user_id} ---")
    start = time.time()

    # Both queries start at the same time!
    user, posts = await asyncio.gather(
        fetch_user(user_id),
        fetch_user_posts(user_id),
    )

    elapsed = time.time() - start
    profile = {**user, "posts": posts}

    print(f"\n  Profile  : {profile}")
    print(f"  ⏱  Time  : {elapsed:.1f}s  (would be 2.5s sequentially)")
    return profile


asyncio.run(fetch_user_profile(42))

--- Fetching profile for user 42 ---
  [DB] Fetching user 42...
  [DB] Fetching posts for user 42...

  Profile  : {'id': 42, 'name': 'User_42', 'posts': [{'id': 1, 'title': 'Post by User_42'}, {'id': 2, 'title': 'Another post by User_42'}]}
  ⏱  Time  : 1.5s  (would be 2.5s sequentially)


{'id': 42,
 'name': 'User_42',
 'posts': [{'id': 1, 'title': 'Post by User_42'},
  {'id': 2, 'title': 'Another post by User_42'}]}

---
## Example 4: `asyncio.create_task()` — Background Work

`create_task()` schedules a coroutine to run **in the background** — your code continues immediately without waiting for it.

This is useful when you want to kick off a non-critical job (like sending a notification) while continuing other work.

In [6]:
async def send_notification(message: str):
    """Simulate sending a notification (slow — takes 2s)."""
    print(f"  📨 Sending notification: '{message}'...")
    await asyncio.sleep(2)
    print(f"  ✅ Notification sent: '{message}'")


async def process_order(order_id: int):
    """Process order AND send notification concurrently using create_task."""
    print(f"--- Processing Order #{order_id} ---")
    start = time.time()

    # 🔑 create_task() schedules the coroutine immediately — we DON'T await yet.
    #    The notification starts running in the background right now.
    notification_task = asyncio.create_task(
        send_notification(f"Order #{order_id} received!")
    )

    # While the notification is being sent, we process the order in parallel
    print(f"  ⚙️  Processing order #{order_id} (takes 1s)...")
    await asyncio.sleep(1)
    print(f"  ✅ Order #{order_id} processed!")

    # NOW we wait for the background notification to finish
    await notification_task

    elapsed = time.time() - start
    print(f"\n  ⏱  Total : {elapsed:.1f}s  (notification + processing overlapped)")


asyncio.run(process_order(101))

--- Processing Order #101 ---
  ⚙️  Processing order #101 (takes 1s)...
  📨 Sending notification: 'Order #101 received!'...
  ✅ Order #101 processed!
  ✅ Notification sent: 'Order #101 received!'

  ⏱  Total : 2.0s  (notification + processing overlapped)


---
## Summary: `gather()` vs `create_task()`

| | `asyncio.gather()` | `asyncio.create_task()` |
|---|---|---|
| **Use when** | You need all results before continuing | You want to fire-and-forget or do other work first |
| **Returns** | List of results (in order) | A `Task` object |
| **Awaiting** | Awaits all at once | You choose when to `await` |
| **Cancel** | Cancels all if one fails | Tasks can be cancelled individually |

---
## Example 5: Run All Examples Together

Execute this cell to run the full demo from start to finish:

In [7]:
async def main():
    print("=" * 50)
    print("  PYTHON ASYNC EXAMPLES — FULL RUN")
    print("=" * 50)

    await run_sequential()          # ~6s
    await run_concurrent()          # ~2s
    await fetch_user_profile(42)    # ~1.5s
    await process_order(101)        # ~2s

    print("\n" + "=" * 50)
    print("  ALL EXAMPLES COMPLETE! 🎉")
    print("=" * 50)


asyncio.run(main())

  PYTHON ASYNC EXAMPLES — FULL RUN
--- Sequential Execution ---
  [ASYNC] Task-A starting...
  [ASYNC] Task-A finished after 2s
  [ASYNC] Task-B starting...
  [ASYNC] Task-B finished after 2s
  [ASYNC] Task-C starting...
  [ASYNC] Task-C finished after 2s

  Results : Task-A_result, Task-B_result, Task-C_result
  ⏱  Total : 6.0s  (expected ~6s)
--- Concurrent Execution (asyncio.gather) ---
  [ASYNC] Task-A starting...
  [ASYNC] Task-B starting...
  [ASYNC] Task-C starting...
  [ASYNC] Task-A finished after 2s
  [ASYNC] Task-B finished after 2s
  [ASYNC] Task-C finished after 2s

  Results : Task-A_result, Task-B_result, Task-C_result
  ⏱  Total : 2.0s  (expected ~2s — 3× faster!)
--- Fetching profile for user 42 ---
  [DB] Fetching user 42...
  [DB] Fetching posts for user 42...

  Profile  : {'id': 42, 'name': 'User_42', 'posts': [{'id': 1, 'title': 'Post by User_42'}, {'id': 2, 'title': 'Another post by User_42'}]}
  ⏱  Time  : 1.5s  (would be 2.5s sequentially)
--- Processing Order 

---
## Practice Exercises

Try these in new cells below:

1. **Timing comparison**: Run `run_sequential()` and `run_concurrent()` inside a single `main()` and print the combined total.
2. **Add a 4th task**: Add `async_task("Task-D", 3)` to `run_concurrent()` — does the total time increase?
3. **Error handling**: What happens if one task raises an exception inside `asyncio.gather()`? Try it!
   ```python
   async def bad_task():
       await asyncio.sleep(1)
       raise ValueError("Something went wrong!")
   ```

In [ ]:
# Your practice code here ✍️
